# IAEA3d

### This is acomputation of the IAEA3d benchmark ![](./IAEA3D.png)

In [ ]:
import inspect

import numpy as np
from IPython.display import Code
from numba.np.unsafe.ndarray import *  # noqa: F403

from dorban.examples import anl_11_iaea3d
from dorban.finite_differences.solve_finite_difference import solve_k_diffusion

## The problem
The materials, the geometry, the cores and the post-processing of the fission source are defined in `dorban.examples.anl_11_iaea3d`, which the benchmark tests use too:

In [ ]:
Code(inspect.getsource(anl_11_iaea3d), language="python")

## Defining the refined core
The refinement is done by hand (`refined_geometry`) and not using the refinement tools, in order to save memory and run time.

In [ ]:
subcells = 2
spliting = anl_11_iaea3d.splitting(subcells)
spliting

In [ ]:
core = anl_11_iaea3d.fd_core(subcells)

## Solving the diffusion equation, reference eigenvalue is 1.02909

In [ ]:
k,flux=solve_k_diffusion(core,rtol_vector = 1e-8,
                      lin_rtol=1e-10)

In [ ]:
print(f"k = {k:.5f}, reference {anl_11_iaea3d.REFERENCE_K}")

## Computing the fission_source

In [ ]:
fission_source = anl_11_iaea3d.thermal_fission_source(flux, core.isotopes)
fuel = [i for i, iso in enumerate(core.isotopes) if iso.isfissile]
core_source = fission_source[fuel]

## Computing the ppf

In [ ]:
np.max(core_source)/np.mean(core_source)

## Computing the radial power distribution

In [ ]:
intermidte_geo, rsource = anl_11_iaea3d.fd_fission_source_by_level(fission_source, subcells)

In [ ]:
radial = np.mean(rsource,axis=1)
radial = radial/[intermidte_geo.volumes[i] for i in range(69)]
radial = radial[radial.nonzero()]
radial/np.mean(radial)

## Computing the axial power distribution

In [ ]:
axial = np.mean(rsource,axis=0)
17.026 * axial/np.sum(axial)

## Compare NEM with finite differences

In [ ]:
subcells = 1
spliting = anl_11_iaea3d.splitting(subcells)
spliting

In [ ]:
from dorban.nem.solve_nem import NEMSettings
from dorban.solve_equation import solve_k

nem_system = anl_11_iaea3d.nem_core(subcells)
nem_k, nem_flux = solve_k(nem_system, NEMSettings(split=None))

In [ ]:
print(f"NEM k = {nem_k:.5f}, finite differences k = {k:.5f}")

## compare axial power distribution

In [ ]:
nem_flux=nem_flux/np.sum(nem_flux)

In [ ]:
nem_fission_source = anl_11_iaea3d.thermal_fission_source(nem_flux, nem_system.isotopes)
fuel = [i for i, iso in enumerate(nem_system.isotopes) if iso.isfissile]
nem_core_source = nem_fission_source[fuel]

In [ ]:
intermidte_geo, nem_rsource = anl_11_iaea3d.nem_fission_source_by_level(nem_fission_source)

In [ ]:
nem_axial = np.mean(nem_rsource,axis=0)
17.026 * nem_axial/np.sum(nem_axial)

In [ ]:
nem_normalized_axial=(nem_axial/np.sum(nem_axial))[1:-1]
normalized_axial=(axial/np.sum(axial))[1:-1]
largest = np.max(np.abs((nem_normalized_axial-normalized_axial)/normalized_axial))
print(f"largest relative difference of the axial power, NEM against finite differences: {largest:.1%}")